In [84]:
import pandas as pd
import numpy as np

In [85]:
df = pd.read_csv("../datasets/LeesmateriaalCSV.csv", sep=";")

In [86]:
#onnodige kolommen verwijderen
df = df.drop(columns=['Unnamed: 3', 'Themas_boek', 'materiaal_id', 'bron_url', 'interne_batchcode', 'gegevensstatus', 'genre_kopie', 'woordenaantal_omschrijving', 'aandeel_lange_woorden_pct', 'aantal_zinnen', 'administratieve_prioriteit'])

In [87]:
#Type materiaal normaliseren
mask = df["Type materiaal"].str.match(r"(?i)^boek\s*-", na=False)
df.loc[mask, "Type materiaal"] = "Boek"
df.loc[df["Type materiaal"] == "boek", "Type materiaal"] = "Boek"
df.loc[df["Type materiaal"] == "digitale krant", "Type materiaal"] = "krant (digitaal)"

In [88]:
#Taal normaliseren
df["taal"] = "NL"

In [89]:
#Aantal paginas naar int type
df.loc[df["aantal_paginas"].isna(), "aantal_paginas"] = -1
df["aantal_paginas"] = df["aantal_paginas"].astype(int)
df.loc[df["aantal_paginas"] < 1, "aantal_paginas"] = np.nan

In [90]:
#Doelgroep normaliseren
df.loc[df["doelgroep"] == "mbo", "doelgroep"] = "MBO"
df.loc[df["doelgroep"] == "mbo / nt2", "doelgroep"] = "MBO/NT2"

In [91]:
#Doelgroepen combineren
def combineer_doelgroepen(row):
    groepen = []

    for waarde in row.dropna():
        for groep in str(waarde).split("/"):
            groep = groep.strip()

            if groep != "" and groep not in groepen:
                groepen.append(groep)

    volgorde = {"MBO": 0, "NT2": 1}
    groepen.sort(key=lambda groep: volgorde.get(groep, 2))

    return "/".join(groepen)

df["doelgroep"] = df[["doelgroep", "doelgroep_extra"]].apply(
    combineer_doelgroepen, axis=1
)
df.loc[df["doelgroep"] == "", "doelgroep"] = np.nan

df = df.drop(["doelgroep_extra"], axis=1)

In [92]:
df.columns.tolist()

['Titel',
 'Auteur',
 'Korte omschrijving',
 'Type materiaal',
 'Niveau',
 'genre',
 'onderwerp',
 'trefwoorden',
 'taal',
 'aantal_paginas',
 'geschatte_leestijd_min',
 'doelgroep',
 'gem_woordlengte',
 'gem_zinslengte_woorden',
 'leescomplexiteit_score',
 'interessecluster',
 'uitgever',
 'publicatiejaar',
 'beschikbaar_formaat',
 'cataloguscode',
 'Gemiddelde beoordeling']

In [93]:
#onrealistische waarden in gem_woordlengte vervangen door NaN
df.loc[df["gem_woordlengte"] > 10, ["gem_woordlengte"]] = np.nan

In [94]:
#interessecluster normaliseren
df.loc[df["interessecluster"] == 'VERHALEN', ["interessecluster"]] = 'verhalen'

In [95]:
#onrealistische en onbekende jaartallen in publicatiejaar vervangen door 'onbekend'
df.loc[df["publicatiejaar"].isna(), ["publicatiejaar"]] = 'onbekend'
df.loc[df["publicatiejaar"] == '2099', ["publicatiejaar"]] = 'onbekend'

In [83]:
df["beschikbaar_formaat"].unique().tolist()

['pdf', 'digitaal', 'online', 'papier', nan]

In [96]:
#beschikbaar_formaat normaliseren
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.lower()

In [97]:
df[["Gemiddelde beoordeling"]][df["Gemiddelde beoordeling"].isna()]

,Gemiddelde beoordeling
183,NaN
184,NaN
185,NaN
186,NaN
187,NaN
...,...
278,NaN
279,NaN
280,NaN
281,NaN


In [99]:
df["Gemiddelde beoordeling"].unique().tolist()

[3.4,
 2.5,
 3.8,
 3.9,
 3.1,
 3.0,
 3.6,
 3.5,
 2.9,
 4.0,
 3.7,
 2.8,
 3.2,
 3.3,
 2.6,
 2.3,
 2.7,
 2.2,
 nan]